# 08 · Administrative boundaries — geoBoundaries

**Source:** geoBoundaries (William & Mary geoLab), open release. Cite: Runfola, D. et al. (2020)
geoBoundaries: A global database of political administrative boundaries. *PLoS ONE* 15(4): e0231866.

- **ADM2 (districts, 736):** boundary year 2021, from Pathways Data Pvt. Ltd. / lgdirectory.gov.in — ODbL 1.0
- **ADM1 (states / UTs, 36):** boundary year 2011, from DataMeet / Election Commission of India — CC BY 2.5 IN

Files already saved are not downloaded again.

## Imports and settings

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))
import json

import requests

OUT = PROJECT / "data" / "raw" / "boundaries"
API = "https://www.geoboundaries.org/api/current/gbOpen/IND/{}/"   # geoBoundaries open release

In [ ]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio"], check=True)

### `download()`

In [2]:
def download(level):
    """Ask the geoBoundaries API for the release, then save the GeoJSON and its metadata file once."""
    info = requests.get(API.format(level), timeout=60).json()
    print(level, "| year:", info["boundaryYearRepresented"], "| units:", info["admUnitCount"],
          "| licence:", info["boundaryLicense"], "| built:", info["buildDate"])
    gj = info["gjDownloadURL"]
    for url, f in [(gj, OUT / f"geoBoundaries-IND-{level}.geojson"),
                   (gj.replace(".geojson", "-metaData.txt"), OUT / f"geoBoundaries-IND-{level}-metaData.txt")]:
        if f.exists():
            print("  already saved:", f.name)
            continue
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        f.write_bytes(r.content)
        print("  saved:", f.name)

## Run

In [3]:
OUT.mkdir(parents=True, exist_ok=True)
for level in ["ADM1", "ADM2"]:
    download(level)

ADM1 | year: 2011 | units: 36 | licence: Creative Commons Attribution 2.5 India (CC BY 2.5 IN) | built: Dec 12, 2023
  already saved: geoBoundaries-IND-ADM1.geojson
  already saved: geoBoundaries-IND-ADM1-metaData.txt


ADM2 | year: 2021 | units: 736 | licence: Open Data Commons Open Database License 1.0 | built: Dec 12, 2023
  already saved: geoBoundaries-IND-ADM2.geojson
  already saved: geoBoundaries-IND-ADM2-metaData.txt


## Check

In [4]:
# the metadata's admUnitCount is compared with the features actually in the saved file
for level in ["ADM1", "ADM2"]:
    n = len(json.loads((OUT / f"geoBoundaries-IND-{level}.geojson").read_text(encoding="utf-8"))["features"])
    meta = requests.get(API.format(level), timeout=60).json()["admUnitCount"]
    print(f"{level}: {n} features in file | metadata admUnitCount {meta}")

ADM1: 36 features in file | metadata admUnitCount 36


ADM2: 735 features in file | metadata admUnitCount 736
